In [ ]:
# SARIMAX model:
# - Date sync to Cambridge end date
# - 95% confidence intervals
# - Rolling backtest (104 train, 4 forecast, step 4)
# - partial block at the end


import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")




CAMBRIDGE_CSV = "/content/cambridge_weekly.csv"
WEATHER_CSV = "/content/weatherdataboston_weekly.csv"

START_DATE = "2020-01-01"
TRAIN_WEEKS = 104
HORIZON = 4
STEP = 4

ORDER = (1, 1, 1)
SEASONAL_ORDER = (0, 1, 1, 52)
MAXITER = 60

ALLOW_PARTIAL_LAST_BLOCK = True

OUT_DIR = Path("/content/sarimax_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = OUT_DIR / "sarimax_cambridge_as_boston_predictions.csv"
BLOCK_PATH = OUT_DIR / "sarimax_cambridge_as_boston_block_metrics.csv"
SUMMARY_PATH = OUT_DIR / "sarimax_cambridge_as_boston_summary.txt"
AUDIT_PATH = OUT_DIR / "date_sync_mismatches.csv"





def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))





cam = pd.read_csv(CAMBRIDGE_CSV, parse_dates=["week_start", "week_end"])
wx = pd.read_csv(WEATHER_CSV, parse_dates=["week_start", "week_end"])

required_cam = {"week_start", "weekly_total_volume"}
required_wx = {"week_start", "tavg_weekly_avg", "prcp_weekly_total"}

missing_cam = required_cam - set(cam.columns)
missing_wx = required_wx - set(wx.columns)
if missing_cam:
    raise ValueError(f"Missing Cambridge columns: {missing_cam}")
if missing_wx:
    raise ValueError(f"Missing Weather columns: {missing_wx}")

cam = cam.sort_values("week_start").reset_index(drop=True)
wx = wx.sort_values("week_start").reset_index(drop=True)

cam_end = cam["week_start"].max()
cam = cam[(cam["week_start"] >= START_DATE) & (cam["week_start"] <= cam_end)].copy()
wx = wx[(wx["week_start"] >= START_DATE) & (wx["week_start"] <= cam_end)].copy()

# check dates and make sure to sync
cam_dates = set(cam["week_start"])
wx_dates = set(wx["week_start"])
cam_not_in_wx = sorted(cam_dates - wx_dates)
wx_not_in_cam = sorted(wx_dates - cam_dates)

audit_rows = []
for d in cam_not_in_wx:
    audit_rows.append({"week_start": d, "missing_from": "weather"})
for d in wx_not_in_cam:
    audit_rows.append({"week_start": d, "missing_from": "cambridge"})

audit_df = pd.DataFrame(audit_rows) if audit_rows else pd.DataFrame(columns=["week_start", "missing_from"])
if len(audit_df):
    audit_df = audit_df.sort_values("week_start")
audit_df.to_csv(AUDIT_PATH, index=False)





df = cam.merge(
    wx[["week_start", "tavg_weekly_avg", "prcp_weekly_total"]],
    on="week_start",
    how="inner"
).sort_values("week_start").reset_index(drop=True)

df = df.dropna(subset=["weekly_total_volume", "tavg_weekly_avg", "prcp_weekly_total"]).reset_index(drop=True)

if len(df) < TRAIN_WEEKS + 1:
    raise ValueError(f"Not enough synced rows ({len(df)}). Need > {TRAIN_WEEKS}.")

synced_start = df["week_start"].min()
synced_end = df["week_start"].max()

print("Cambridge end date:", cam_end.date())
print("Synced start/end:", synced_start.date(), "to", synced_end.date())
print("Cam weeks:", len(cam), "Weather weeks:", len(wx), "Synced weeks:", len(df))
print("Date mismatches:", len(audit_df))

# rolling SARIMAX with a 95% CI
pred_rows = []
block_rows = []

max_start = len(df) - TRAIN_WEEKS - HORIZON
if max_start < 0:
    raise ValueError("Not enough rows for one full block.")

starts = list(range(0, max_start + 1, STEP))

for block_id, s in enumerate(starts, start=1):
    train = df.iloc[s:s + TRAIN_WEEKS]
    test = df.iloc[s + TRAIN_WEEKS:s + TRAIN_WEEKS + HORIZON]

    y_train = train["weekly_total_volume"].astype(float)
    x_train = train[["tavg_weekly_avg", "prcp_weekly_total"]].astype(float)
    x_test = test[["tavg_weekly_avg", "prcp_weekly_total"]].astype(float)

    fit = SARIMAX(
        y_train,
        exog=x_train,
        order=ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False, maxiter=MAXITER)

    fc = fit.get_forecast(steps=len(test), exog=x_test)
    y_pred = np.asarray(fc.predicted_mean, dtype=float)
    ci = fc.conf_int(alpha=0.05)

    lower_col = [c for c in ci.columns if "lower" in c.lower()][0]
    upper_col = [c for c in ci.columns if "upper" in c.lower()][0]
    y_lo = np.asarray(ci[lower_col], dtype=float)
    y_hi = np.asarray(ci[upper_col], dtype=float)

    y_true = test["weekly_total_volume"].to_numpy(dtype=float)

    block_rows.append({
        "block": block_id,
        "train_start": train["week_start"].iloc[0],
        "train_end": train["week_start"].iloc[-1],
        "pred_start": test["week_start"].iloc[0],
        "pred_end": test["week_start"].iloc[-1],
        "weeks_in_block": len(test),
        "rmse": rmse(y_true, y_pred),
        "mae": mae(y_true, y_pred),
    })

    for d, a, p, lo, hi in zip(test["week_start"], y_true, y_pred, y_lo, y_hi):
        pred_rows.append({
            "week_start": d,
            "actual": float(a),
            "predicted": float(p),
            "ci95_lower": float(lo),
            "ci95_upper": float(hi),
            "inside_ci95": bool(lo <= a <= hi),
            "abs_error": float(abs(a - p)),
            "block": block_id,
        })

# Partial final block reminder
# making sure that we are allowed a partial block - ensures that the data that syncs up from 2026 isn't cut off as a result of the 4-week blocks
if ALLOW_PARTIAL_LAST_BLOCK:
    last_full_pred_end_idx = starts[-1] + TRAIN_WEEKS + HORIZON - 1
    tail_start_idx = last_full_pred_end_idx + 1
    if tail_start_idx < len(df):
        remaining = len(df) - tail_start_idx
        train_end_idx = tail_start_idx - 1
        train_start_idx = train_end_idx - TRAIN_WEEKS + 1
        if train_start_idx >= 0 and remaining > 0:
            train = df.iloc[train_start_idx:train_end_idx + 1]
            test = df.iloc[tail_start_idx:len(df)]

            y_train = train["weekly_total_volume"].astype(float)
            x_train = train[["tavg_weekly_avg", "prcp_weekly_total"]].astype(float)
            x_test = test[["tavg_weekly_avg", "prcp_weekly_total"]].astype(float)

            fit = SARIMAX(
                y_train,
                exog=x_train,
                order=ORDER,
                seasonal_order=SEASONAL_ORDER,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False, maxiter=MAXITER)

            fc = fit.get_forecast(steps=len(test), exog=x_test)
            y_pred = np.asarray(fc.predicted_mean, dtype=float)
            ci = fc.conf_int(alpha=0.05)

            lower_col = [c for c in ci.columns if "lower" in c.lower()][0]
            upper_col = [c for c in ci.columns if "upper" in c.lower()][0]
            y_lo = np.asarray(ci[lower_col], dtype=float)
            y_hi = np.asarray(ci[upper_col], dtype=float)

            y_true = test["weekly_total_volume"].to_numpy(dtype=float)
            partial_block_id = len(block_rows) + 1

            block_rows.append({
                "block": partial_block_id,
                "train_start": train["week_start"].iloc[0],
                "train_end": train["week_start"].iloc[-1],
                "pred_start": test["week_start"].iloc[0],
                "pred_end": test["week_start"].iloc[-1],
                "weeks_in_block": len(test),
                "rmse": rmse(y_true, y_pred),
                "mae": mae(y_true, y_pred),
            })

            for d, a, p, lo, hi in zip(test["week_start"], y_true, y_pred, y_lo, y_hi):
                pred_rows.append({
                    "week_start": d,
                    "actual": float(a),
                    "predicted": float(p),
                    "ci95_lower": float(lo),
                    "ci95_upper": float(hi),
                    "inside_ci95": bool(lo <= a <= hi),
                    "abs_error": float(abs(a - p)),
                    "block": partial_block_id,
                })

pred_df = pd.DataFrame(pred_rows).sort_values("week_start").reset_index(drop=True)
block_df = pd.DataFrame(block_rows)

overall_rmse = rmse(pred_df["actual"], pred_df["predicted"])
overall_mae = mae(pred_df["actual"], pred_df["predicted"])
coverage95 = float(pred_df["inside_ci95"].mean() * 100.0)

pred_df.to_csv(PRED_PATH, index=False)
block_df.to_csv(BLOCK_PATH, index=False)


summary_lines = [
    "SARIMAX Rolling Summary (Synced to Cambridge End + 95% CI)",
    f"Start date: {START_DATE}",
    f"Cambridge end date used: {cam_end.date()}",
    f"Synced start date: {synced_start.date()}",
    f"Synced end date: {synced_end.date()}",
    f"Train weeks: {TRAIN_WEEKS}",
    f"Horizon: {HORIZON}",
    f"Step: {STEP}",
    f"Allow partial last block: {ALLOW_PARTIAL_LAST_BLOCK}",
    f"Order: {ORDER}",
    f"Seasonal order: {SEASONAL_ORDER}",
    f"Blocks run: {len(block_df)}",
    f"Predicted rows: {len(pred_df)}",
    f"First predicted week: {pred_df['week_start'].min().date()}",
    f"Last predicted week: {pred_df['week_start'].max().date()}",
    f"Overall RMSE: {overall_rmse:.2f}",
    f"Overall MAE: {overall_mae:.2f}",
    f"95% CI empirical coverage (%): {coverage95:.2f}",
    f"Date mismatch rows: {len(audit_df)}",
]
SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

print("\n=== DONE ===")
print("First predicted week:", pred_df["week_start"].min().date())
print("Last predicted week:", pred_df["week_start"].max().date())
print("Overall RMSE:", round(overall_rmse, 2))
print("Overall MAE:", round(overall_mae, 2))
print("95% CI coverage (%):", round(coverage95, 2))
print("\nSaved:")
print(PRED_PATH)
print(BLOCK_PATH)
print(SUMMARY_PATH)
print(AUDIT_PATH)

# downloading the files
from google.colab import files
files.download('/content/sarimax_outputs/sarimax_cambridge_as_boston_predictions.csv')
files.download('/content/sarimax_outputs/sarimax_cambridge_as_boston_block_metrics.csv')
files.download('/content/sarimax_outputs/sarimax_cambridge_as_boston_summary.txt')

Cambridge end date: 2026-02-02
Synced start/end: 2020-01-27 to 2026-02-02
Cam weeks: 315 Weather weeks: 318 Synced weeks: 315
Date mismatches: 3

=== DONE ===
First predicted week: 2022-01-24
Last predicted week: 2026-02-02
Overall RMSE: 132386.37
Overall MAE: 93357.66
95% CI coverage (%): 93.36

Saved:
/content/sarimax_outputs/sarimax_cambridge_as_boston_predictions.csv
/content/sarimax_outputs/sarimax_cambridge_as_boston_block_metrics.csv
/content/sarimax_outputs/sarimax_cambridge_as_boston_summary.txt
/content/sarimax_outputs/date_sync_mismatches.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# RISK ANALYSIS VISUALS - two graphs

import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import ScalarFormatter

pred_path = Path('/Users/yiyuanemmazhou/Downloads/sarimax_cambridge_as_boston_predictions-2.csv')
out_dir = Path('/Users/yiyuanemmazhou/Documents/New Project')

plot1 = out_dir / 'sarimax_actual_vs_predicted_weekly.png'
plot2 = out_dir / 'sarimax_uncertainty_rmse_by_block.png'

pred = pd.read_csv(pred_path, parse_dates=['week_start']).sort_values('week_start')

block_meta = pred.groupby('block', as_index=False).agg(
    pred_start=('week_start', 'min'),
    pred_end=('week_start', 'max'),
    weeks_in_block=('week_start', 'count')
)
rmse_by_block = pred.groupby('block').apply(
    lambda g: float(np.sqrt(np.mean((g['actual'] - g['predicted']) ** 2)))
).reset_index(name='rmse')
block = block_meta.merge(rmse_by_block, on='block').sort_values('pred_start')

mean_uncertainty = float(block['rmse'].mean())

# calendar-based ticks (the ticks aren't going to be perfectly spaced out with the major year ticks)
year_locator = mdates.YearLocator()
year_fmt = mdates.DateFormatter('%Y')
minor_locator = mdates.WeekdayLocator(byweekday=mdates.MO, interval=4)

def plain_y_axis(ax):
    fmt = ScalarFormatter(useOffset=False)
    fmt.set_scientific(False)
    ax.yaxis.set_major_formatter(fmt)
    ax.ticklabel_format(axis='y', style='plain', useOffset=False)

# Graph 1: Actual volume vs Predicted volume
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(
    pred['week_start'],
    pred['actual'],
    color='#1f77b4',
    linewidth=2.0,
    label='Actual weekly volume (vehicle count)'
)
ax.plot(
    pred['week_start'],
    pred['predicted'],
    color='#d62728',
    linewidth=2.0,
    label='SARIMAX predicted weekly volume (vehicle count)'
)
ax.fill_between(
    pred['week_start'],
    pred['actual'],
    pred['predicted'],
    color='#9ecae1',
    alpha=0.25,
    label='Gap between lines'
)

ax.set_title('Weekly Aggregated Volume: Actual vs SARIMAX Prediction')
ax.set_xlabel('Year (minor ticks every 4 weeks)')
ax.set_ylabel('Weekly Volume (vehicle count)')

ax.xaxis.set_major_locator(year_locator)
ax.xaxis.set_major_formatter(year_fmt)
ax.xaxis.set_minor_locator(minor_locator)
ax.tick_params(axis='x', which='major', length=8)
ax.tick_params(axis='x', which='minor', length=3)

plain_y_axis(ax)

ax.grid(True, which='major', alpha=0.25)
ax.legend(loc='upper left', frameon=True)
fig.tight_layout()
fig.savefig(plot1, dpi=220)
plt.close(fig)

# Graph 2: Uncertainty (RMSE by block) with a mean threshold line
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(
    block['pred_start'],
    block['rmse'],
    color='#9467bd',
    marker='o',
    markersize=3.2,
    linewidth=1.8,
    label='Block RMSE (vehicle count)'
)
ax.axhline(
    mean_uncertainty,
    color='#ff7f0e',
    linestyle='--',
    linewidth=2,
    label=f'Mean uncertainty threshold = {mean_uncertainty:,.0f} vehicle count'
)

ax.set_title('Forecast Uncertainty by 4-Week Block (RMSE)')
ax.set_xlabel('Year (minor ticks every 4 weeks)')
ax.set_ylabel('Uncertainty (RMSE, vehicle count)')

ax.xaxis.set_major_locator(year_locator)
ax.xaxis.set_major_formatter(year_fmt)
ax.xaxis.set_minor_locator(minor_locator)
ax.tick_params(axis='x', which='major', length=8)
ax.tick_params(axis='x', which='minor', length=3)

plain_y_axis(ax)

ax.grid(True, which='major', alpha=0.25)
ax.legend(loc='upper left', frameon=True)
fig.tight_layout()
fig.savefig(plot2, dpi=220)
plt.close(fig)

print('updated', plot1)
print('updated', plot2)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/yiyuanemmazhou/Downloads/sarimax_cambridge_as_boston_predictions-2.csv'

In [ ]:
from pathlib import Path
import os

print("Current working directory:", os.getcwd())

out_dir = Path.cwd() / "sarimax_outputs"
print("Expected output folder:", out_dir)
print("Folder exists:", out_dir.exists())

if out_dir.exists():
    print("\nFiles inside:")
    for p in out_dir.glob("*"):
        print("-", p.resolve())


In [ ]:
from google.colab import files
files.upload()

Saving weatherdataboston_weekly.csv to weatherdataboston_weekly.csv


{'weatherdataboston_weekly.csv': b'week_start,week_end,tavg_weekly_avg,prcp_weekly_total\n2018-01-01,2018-01-07,-10.4,34.3\n2018-01-08,2018-01-14,2.0142857142857142,47.2\n2018-01-15,2018-01-21,-1.242857142857143,5.3\n2018-01-22,2018-01-28,1.9714285714285715,34.3\n2018-01-29,2018-02-04,-1.4285714285714288,18.6\n2018-02-05,2018-02-11,0.8857142857142858,47.2\n2018-02-12,2018-02-18,3.271428571428572,13.7\n2018-02-19,2018-02-25,7.214285714285714,19.8\n2018-02-26,2018-03-04,6.057142857142857,58.699999999999996\n2018-03-05,2018-03-11,1.9000000000000001,33.3\n2018-03-12,2018-03-18,0.35714285714285715,30.3\n2018-03-19,2018-03-25,1.442857142857143,4.4\n2018-03-26,2018-04-01,6.071428571428571,2.6\n2018-04-02,2018-04-08,3.642857142857143,25.400000000000002\n2018-04-09,2018-04-15,5.5,2.5\n2018-04-16,2018-04-22,7.1000000000000005,45.699999999999996\n2018-04-23,2018-04-29,11.757142857142856,38.6\n2018-04-30,2018-05-06,17.357142857142858,12.2\n2018-05-07,2018-05-13,12.014285714285716,5.3\n2018-05-14,2

In [ ]:
from google.colab import files
files.upload()

Saving cambridge_weekly.csv to cambridge_weekly.csv


{'cambridge_weekly.csv': b'week_start,week_end,weekly_total_volume,weekly_volume_index\n2020-01-27,2020-02-02,402916,0.23575658279303133\n2020-02-03,2020-02-09,1721182,1.007108148311001\n2020-02-10,2020-02-16,1691560,0.9897755492196391\n2020-02-17,2020-02-23,1612749,0.9436611927619616\n2020-02-24,2020-03-01,1793647,1.0495092958754983\n2020-03-02,2020-03-08,1788208,1.0463267961638678\n2020-03-09,2020-03-15,1558986,0.9122030695782167\n2020-03-16,2020-03-22,928198,0.5431126801500216\n2020-03-23,2020-03-29,654444,0.38293212746429184\n2020-03-30,2020-04-05,624308,0.36529877366585545\n2020-04-06,2020-04-12,636894,0.37266316810795524\n2020-04-13,2020-04-19,647438,0.3788327354842067\n2020-04-20,2020-04-26,672800,0.39367269828736384\n2020-04-27,2020-05-03,725296,0.424389466969429\n2020-05-04,2020-05-10,765022,0.4476341780457724\n2020-05-11,2020-05-17,795776,0.46562913964376523\n2020-05-18,2020-05-24,852964,0.49909131899819115\n2020-05-25,2020-05-31,913570,0.5345534586420734\n2020-06-01,2020-06-